Train the model

In [1]:
!pip uninstall -y torch torchvision torchaudio xformers
!pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 torchaudio==2.4.0+cu121 --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 50.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 56.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 43.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 63.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 826.7

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

In [ ]:
import random

companies = [
    "Chase",
    "Bank of America",
    "Capital One",
    "Wells Fargo",
    "Amazon",
    "UPS",
    "FedEx",
    "Delta",
    "United Airlines",
    "CVS",
    "Walgreens",
    "Microsoft"
]

names = [
    "John Smith",
    "Sarah Johnson",
    "Emily Davis",
    "Michael Brown",
    "David Wilson"
]

titles = [
    "",
    "Mr. ",
    "Mrs. ",
    "Ms. ",
    "Dr. "
]

products = [
    "iPhone 16",
    "MacBook Air",
    "prescription",
    "gaming laptop",
    "package"
]

dates = [
    "August 5",
    "August 12",
    "Tomorrow",
    "Monday",
    "September 1"
]

times = [
    "9:00 AM",
    "2:30 PM",
    "4:00 PM",
    "10:15 AM"
]

money = [
    "$12.99",
    "$84.51",
    "$215.00",
    "$1,204.18"
]
cities = [
    "New York",
    "Albany",
    "Chicago",
    "Seattle",
    "Boston"
]

years = [
    "2024",
    "2025",
    "2026"
]

numbers = [
    "847392",
    "194857",
    "560123",
    "932184"
]

In [ ]:
from datasets import Dataset
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/scamdetector/dataset.csv")

dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [ ]:
def fill(text):
    if pd.isna(text):
        return text
    text = text.replace("[Company]", random.choice(companies))
    text = text.replace("[Name]", random.choice(names))
    text = text.replace("[Title]", random.choice(titles))
    text = text.replace("[Product]", random.choice(products))
    text = text.replace("[Date]", random.choice(dates))
    text = text.replace("[Time]", random.choice(times))
    text = text.replace("[Money]", random.choice(money))
    text = text.replace("[City]", random.choice(cities))
    text = text.replace("[Year]", random.choice(years))
    text = text.replace("[Number]", random.choice(numbers))
    return text

In [ ]:
rows = []
train_df = train_dataset.to_pandas()
for _, row in train_df.iterrows():
    for _ in range(5):
        new_row = row.copy()
        new_row["text"] = fill(row["text"])
        rows.append(new_row)

augmented_train_df = pd.DataFrame(rows)

from datasets import Dataset
train_dataset = Dataset.from_pandas(augmented_train_df)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
!pip install evaluate

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="scam_model",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=50
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.001769,0.001097,1.000000
2,0.000673,0.000453,1.000000
3,0.000496,0.000356,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.026538488967344166, metrics={'train_runtime': 299.7645, 'train_samples_per_second': 32.025, 'train_steps_per_second': 2.002, 'total_flos': 635843513548800.0, 'train_loss': 0.026538488967344166, 'epoch': 3.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/scamdetector/model")
tokenizer.save_pretrained("/content/drive/MyDrive/scamdetector/model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/scamdetector/model/tokenizer_config.json',
 '/content/drive/MyDrive/scamdetector/model/tokenizer.json')

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/scamdetector/model",
    tokenizer="/content/drive/MyDrive/scamdetector/model"
)

classifier(
    """Subject: Your monthly subscription summary for Walmart

Hello Kyle,

This is a friendly reminder that your monthly subscription plan for Walmart renews on 4/11/2022. Your payment method on file will be automatically charged $9.99.

No action is needed on your part. You can manage, update, or cancel your subscription at any time by securely logging into your account settings directly through our official app or website.

If you have any questions or need support, please reach out to our team through the Help Center inside your account dashboard.

Thanks for being a customer!

Best regards,

Customer Support Team"""
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'scam', 'score': 0.9987987279891968}]

In [ ]:
print(df['label'].value_counts())

label
0    400
1    400
Name: count, dtype: int64


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

id2label = {0: "legit", 1: "scam"}
label2id = {"legit": 0, "scam": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)
training_args = TrainingArguments(
    output_dir="scam_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.05,
    warmup_ratio=0.1,
    logging_steps=20,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.002159,0.001534,1.000000
2,0.001157,0.000830,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=800, training_loss=0.06817323831608518, metrics={'train_runtime': 163.4438, 'train_samples_per_second': 39.157, 'train_steps_per_second': 4.895, 'total_flos': 423895675699200.0, 'train_loss': 0.06817323831608518, 'epoch': 2.0})

In [72]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/scamdetector/model",
    tokenizer="/content/drive/MyDrive/scamdetector/model"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'scam', 'score': 0.9987987279891968}]

In [79]:
text = """Subject: Your monthly Netflix membership receipt

Hi [Name],

This is a quick confirmation that we successfully charged your payment method on file for your standard monthly Netflix plan on [Date].

Amount charged: $15.99

Billing period: [Date] to [Date]

Payment method: Visa ending in ••••

No changes have been made to your account. If you ever want to check your billing history, update your payment details, or change your plan, you can securely access those options anytime by logging into your account directly through the official Netflix website or app.

Thanks for being a member!

Best,

The Netflix Team"""

result = classifier(text)[0]

label = result['label']
score = result['score']

if label == 'scam' and score > 0.95:
    text_lower = text.lower()
    if "official app" in text_lower or "no action is needed" in text_lower:
        print("Result: legit (Override: Safe structural indicators found)")
    else:
        print(f"Result: {label} (Score: {score:.4f})")
else:
    print("Result: legit")

### THIS IS WITHOUT THE MAIN KEY WORDS IN THE WEBSITE

Result: scam (Score: 0.9988)
